In [1]:
from SelfCal import PipelineWrapper
from astropy.io import fits
import numpy as np
import glob
from SelfCal.SPHERExUtility import interpolate_array, make_chunk_map, make_chunk_mask, visualize_chunk_map, interp_2d_vertical
import matplotlib.pyplot as plt
import matplotlib as mpl
# Import LogNorm
from tqdm import tqdm
import os
from SelfCal import MakeMap
import gc

In [63]:
detector = 4
config = {}
config['output_dir'] = '/mnt/md124/thomasli/selfcal/outputs/'
config['run_name'] = f'nep_det{detector}_6p2arcsec'
config['resolution_arcsec'] = 6.2

chunk_map = make_chunk_map(detector, interp_factor=10,
                           calibration_file='/home/thomasli/spherex/SSDC_CENTER_all_v20250202.fits',
                          channel_file='/home/thomasli/spherex/spherex_channels.csv')

100%|██████████████████████████████████████████████████████████████████████| 172/172 [00:10<00:00, 16.21it/s]


100%|███████████████████████████████████████████████████████████████| 3458/3458 [00:00<00:00, 1591736.53it/s]

Loading reference frame from: /mnt/md124/thomasli/selfcal/outputs/nep_det1_6p2arcsec/ref.fits


In [ ]:
for ch in [[1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17]]:
    print(f"Processing channel {ch} for detector {detector}")
    
    chunk_valid_mask = make_chunk_mask(ch, interp_factor=10)
    det_valid_mask = chunk_valid_mask[chunk_map]
    cc = PipelineWrapper.Calibrator(config)
    
    num_reproj = len(cc.reproj_list)
    index_full = np.arange(num_reproj)
    index_list1 = np.sort(np.random.choice(index_full, size=num_reproj//2, replace=False))
    index_list2 = index_full[~np.isin(index_full, index_list1)]
    
    cc1 = PipelineWrapper.Calibrator(config)
    
    for i, indices in enumerate([index_list1, index_list2]):
        cc1.reproj_list = [cc.reproj_list[i] for i in indices]
        cc1.exp_idx_list = np.arange(len(cc1.reproj_list))
        cc1.det_idx_list = np.full(len(cc1.reproj_list), 0)
        
        cc1.setup_lsqr(
            apply_mask=True, 
            apply_weight=False, 
            chunk_map=chunk_map, 
            det_valid_mask=det_valid_mask, 
            max_workers=50, 
            outlier_thresh=20.0,
            ignore_list=[],
            )
        
        cc1.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-1, iter_lim=500)
        
        cal_path = cc1.save_calibration(cal_file=f'cal_det{detector}_ch{'-'.join(map(str, ch))}_split{i+1}.h5')
        
        mm1 = PipelineWrapper.Mosaicker(config)
        mm1.reproj_list = cc1.reproj_list
        mm1.exp_idx_list = cc1.exp_idx_list
        mm1.det_idx_list = cc1.det_idx_list
        
        mm1.load_calibration(cal_path=cal_path)
        
        maps = mm1.make_mosaic(
            apply_mask=True, 
            apply_weight=False, 
            chunk_map=chunk_map, 
            det_valid_mask=det_valid_mask, 
            max_workers=50,
            make_std_map=True, 
            apply_sigma_clipping=True, 
            sigma=1.0,
            ignore_list=[21],
            interp_func=interp_2d_vertical
        )
        
        mm1.save_mosaic(mos_file=f'mosaic_det{detector}_ch{"-".join(map(str, ch))}_split{i+1}.fits', overwrite=True)

Processing channel [1] for detector 4


100%|████████████████████████████████████████████████████████████████| 3454/3454 [00:00<00:00, 819639.38it/s]


Loading reference frame from: /mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/ref.fits


100%|████████████████████████████████████████████████████████████████| 3454/3454 [00:00<00:00, 859209.18it/s]

Loading reference frame from: /mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/ref.fits


...

Calibration loaded from /mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/calibration/cal_det4_ch12_split1.h5


Computing sigma_clip map: 100%|██████████████████████████████████████████████| 50/50 [05:30<00:00,  6.60s/it]


Mosaic saved to /mnt/md124/thomasli/selfcal/outputs/nep_det4_6p2arcsec/mosaic/mosaic_det4_ch12_split1.fits


Building A, b matrix: 100%|██████████████████████████████████████████████| 1727/1727 [06:05<00:00,  4.72it/s]


Solving least squares for 81056870 unknowns with 145238527 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 145238527 rows and 81056870 columns
damp = 1.00000000000000e-01   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      500
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   9.874e+02  9.874e+02    1.0e+00  9.3e-02
     1  0.00000e+00   1.893e+02  1.893e+02    1.9e-01  7.1e-01   9.4e+01  1.0e+00
     2  0.00000e+00   1.167e+02  1.167e+02    1.2e-01  2.5e-01   1.3e+02  2.0e+00
     3  0.00000e+00   1.066e+02  1.066e+02    1.1e-01  8.1e-02   1.5e+02  3.1e+00
     4  0.00000e+00   1.053e+02  1.053e+02    1.1e-01  3.2e-02   1.7e+02  4.3e+00
     5  0.00000e+00   1.049e+02  1.049e+02    1.1e-01  2.4e-02   1.9e+02  5.7e+00
     6  0.00000e+00   1.045e+02  1.046e+02    1.1e-01  2.3e-02   2.0e+02  7.9e+00


In [ ]:
print('hi')